# Feature Engineering für Kredit-Scoring

Aus den bereinigten Daten werden hier produktionsreife Features für Training und Submission-Testdaten aufgebaut. Fokus: irrelevante Spalten entfernen, kreditrelevante Features herstellen und beide Splits identisch strukturieren.

## 1. Vorgehen
- Daten laden (train_eda/test_cleaned)
- Globale Variablen & Hilfsfunktionen definieren (ID-Drops, Mapping-Tabellen)
- Feature-Pipeline anwenden: IDs entfernen, Domänen/Relationen berechnen
- Gleiches Spalten-Set für train/test sicherstellen
- Ergebnis als `train_fe.csv` bzw. `test_fe.csv` persistieren

In [1]:
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_csv('../data/train/train_eda.csv')
test_df = pd.read_csv('../data/test/test_cleaned.csv')

print(f'Trainingsdaten: {train_df.shape[0]:,} Zeilen / {train_df.shape[1]} Spalten')
print(f'Testdaten: {test_df.shape[0]:,} Zeilen / {test_df.shape[1]} Spalten')
train_df.head(2)

Trainingsdaten: 72,878 Zeilen / 28 Spalten
Testdaten: 50,000 Zeilen / 27 Spalten


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Unknown,809.98,26.82262,22.1,No,49.6,80.415295,High Spent Small Value Payments,312.494089,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1592.843333,3,...,Good,809.98,31.94496,0.0,No,49.6,118.280222,Low Spent Large Value Payments,284.629162,Good


## 2. Globale Variablen & Hilfsfunktionen
Die Konstanten kapseln Domänenannahmen (z. B. Monatsreihenfolge, Mapping von Kategorien zu Scores) und werden von allen weiteren Schritten genutzt.

In [3]:
DROP_IDENTIFIERS = ['ID', 'Customer_ID', 'Name', 'SSN']

PAY_MIN_MAP = {'Yes': 2, 'No': 1, 'Unknown': 0}

CREDIT_MIX_MAP = {'Bad': 0, 'Unknown': 1, 'Standard': 2, 'Good': 3}

SPENT_MAP = {'High': 2, 'Low': 1}
VALUE_MAP = {'Small': 1, 'Medium': 2, 'Large': 3}

EPS = 1e-6

def clean_loan_list(raw_value):
    'Bricht die Textliste der Kredite in eine eindeutige Python-Liste herunter.'
    'Bsp.: "Studen Loand and Home Equity Loan" --> [Student Loand", "Home Equity Loan"]'
    if pd.isna(raw_value):
        return []
    text = str(raw_value).replace(' and ', ', ')
    return [item.strip() for item in text.split(',') if item.strip()]

def split_payment_behaviour(text):
    'Zerlegt Payment_Behaviour in (Spent-Level, Value-Level, Unknown-Flag).'
    if pd.isna(text) or str(text).lower() == 'unknown':
        return 0, 0, 1
    tokens = str(text).split()
    spent = SPENT_MAP.get(tokens[0], 0) if tokens else 0
    value = VALUE_MAP.get(tokens[3], 0) if len(tokens) >= 4 else 0
    return spent, value, 0

def credit_history_to_months(value):
    'Wandelt dezimale Credit-History-Angaben (z.B. 22.4) in Monate um.'
    if pd.isna(value):
        return np.nan
    years = np.floor(value)
    months = np.round((value - years) * 10)
    return years * 12 + months

In [4]:
combined_for_stats = pd.concat([
    train_df.drop(columns=['Credit_Score']),
    test_df.copy()
], ignore_index=True)

loan_lists_all = combined_for_stats['Type_of_Loan'].apply(clean_loan_list)
flat_loans = [loan for sublist in loan_lists_all for loan in sublist if loan != 'Not Specified']
if flat_loans:
    loan_frequency = pd.Series(flat_loans, dtype='object').value_counts()
    TOP_LOAN_TYPES = loan_frequency.head(5).index.tolist()
else:
    TOP_LOAN_TYPES = []

print('Top Loan-Typen (global):', TOP_LOAN_TYPES if TOP_LOAN_TYPES else 'Keine vorhanden')

Top Loan-Typen (global): ['Credit-Builder Loan', 'Payday Loan', 'Student Loan', 'Home Equity Loan', 'Debt Consolidation Loan']


## 3. Feature-Pipeline
Die Funktion `engineer_features` kapselt sämtliche Drops, Transformationen und neuen Merkmale, damit Train- und Testdaten identisch verarbeitet werden.
- IDs werden entfernt
- Type_of_Loan Count Feature erstellen
- Payment_Behaviour Wörter tokensieren und gewichten
- Payment_of_Min_Amount daraus ein Score erstellen (0 bis 2)
- Credit_Mix daraus ein Score erstellen
- Credit_History als Zahl umwandeln
- Credit_Age_Gap Differenz zwischen Alter und Kreditdauer
- Credit_History_to_Age Anteil der Kreditdauer am Alter
- Zahlungsverzugs-Features
- Zahlreiche Verhältnis-Features, die Schulden/EMI/Investitionen in Relation zum Einkommen setzen.
    - EPS wird zur Vermeidung von Division durch 0 addiert.
- Log-Transformationen bei schiefe Verteilungen

In [5]:
def engineer_features(df):
    'Erstellt eine modellfertige Kopie der Eingabe ohne Zielvariable.'
    result = df.copy()

    drop_cols = [col for col in DROP_IDENTIFIERS if col in result.columns]
    if drop_cols:
        result = result.drop(columns=drop_cols)

    loan_lists = result['Type_of_Loan'].apply(clean_loan_list)
    result['Loan_Types_Count'] = loan_lists.apply(lambda items: len({itm for itm in items if itm != 'Not Specified'}))
    result['Has_Not_Specified_Loan'] = loan_lists.apply(lambda items: int('Not Specified' in items))
    for loan in TOP_LOAN_TYPES:
        col_name = 'LoanType_' + loan.replace(' ', '_')
        result[col_name] = loan_lists.apply(lambda items, loan=loan: int(loan in items))
    result = result.drop(columns=['Type_of_Loan'])

    payment_split = result['Payment_Behaviour'].fillna('Unknown').apply(split_payment_behaviour)
    payment_df = pd.DataFrame(payment_split.tolist(), columns=['Payment_Spent_Level', 'Payment_Value_Level', 'Is_Payment_Behaviour_Unknown'], index=result.index)
    payment_df = payment_df.astype(int)
    result = pd.concat([result.drop(columns=['Payment_Behaviour']), payment_df], axis=1)

    pay_min = result['Payment_of_Min_Amount'].fillna('Unknown')
    result['Pays_Min_Amount_Score'] = pay_min.map(PAY_MIN_MAP).fillna(0).astype(int)
    result['Is_Min_Payment_Unknown'] = (pay_min == 'Unknown').astype(int)
    result = result.drop(columns=['Payment_of_Min_Amount'])

    result['Credit_Mix_Score'] = result['Credit_Mix'].map(CREDIT_MIX_MAP).fillna(1).astype(int)

    credit_age = result['Credit_History_Age'].fillna(0)
    result['Credit_History_Months'] = credit_age.apply(credit_history_to_months)
    result['Credit_Age_Gap'] = (result['Age'] - credit_age).clip(lower=0)
    result['Credit_History_to_Age'] = (credit_age / result['Age'].replace(0, np.nan)).fillna(0)

    result['Delay_Impact'] = result['Delay_from_due_date'] * result['Num_of_Delayed_Payment']
    result['Delay_per_Loan'] = result['Delay_Impact'] / (result['Num_of_Loan'] + 1)
    result['Delayed_Payment_Ratio'] = result['Num_of_Delayed_Payment'] / (result['Num_of_Loan'] + 1)

    result['Debt_To_Income_Ratio'] = result['Outstanding_Debt'] / (result['Annual_Income'] + EPS)
    result['EMI_To_Income_Ratio'] = result['Total_EMI_per_month'] / (result['Monthly_Inhand_Salary'] + EPS)
    result['Invest_To_Income_Ratio'] = result['Amount_invested_monthly'] / (result['Monthly_Inhand_Salary'] + EPS)
    result['Savings_To_Salary_Ratio'] = result['Monthly_Balance'] / (result['Monthly_Inhand_Salary'] + EPS)
    result['Expense_Burden_Ratio'] = (result['Total_EMI_per_month'] + result['Amount_invested_monthly']) / (result['Monthly_Inhand_Salary'] + EPS)
    result['Income_Per_Loan'] = result['Annual_Income'] / (result['Num_of_Loan'] + 1)
    result['Income_Per_Credit_Inquiry'] = result['Annual_Income'] / (result['Num_Credit_Inquiries'] + 1)
    result['Disposable_Income'] = result['Monthly_Inhand_Salary'] - result['Total_EMI_per_month'] - result['Amount_invested_monthly']
    result['Disposable_Income_Ratio'] = result['Disposable_Income'] / (result['Monthly_Inhand_Salary'] + EPS)

    result['Credit_Utilization_x_Delay'] = result['Credit_Utilization_Ratio'] * result['Num_of_Delayed_Payment']
    result['Utilization_per_Card'] = result['Credit_Utilization_Ratio'] / (result['Num_Credit_Card'] + 1)
    result['Accounts_per_Loan'] = (result['Num_Bank_Accounts'] + 1) / (result['Num_of_Loan'] + 1)

    log_features = ['Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Num_Credit_Inquiries', 'Delay_Impact', 'Monthly_Balance']
    for col in log_features:
        result[col + '_log1p'] = np.log1p(result[col].clip(lower=0))

    result = result.replace([np.inf, -np.inf], np.nan)
    return result

In [6]:
train_target = train_df['Credit_Score'].copy()
train_df = engineer_features(train_df.drop(columns=['Credit_Score']))
train_df['Credit_Score'] = train_target.values

test_df = engineer_features(test_df)

feature_cols = [col for col in train_df.columns if col != 'Credit_Score']
missing_in_test = [col for col in feature_cols if col not in test_df.columns]
for col in missing_in_test:
    test_df[col] = 0

extra_in_test = [col for col in test_df.columns if col not in feature_cols]
if extra_in_test:
    test_df = test_df.drop(columns=extra_in_test)

train_df = train_df[feature_cols + ['Credit_Score']]
test_df = test_df[feature_cols]

print(f'Feature-Satz (train): {len(feature_cols)} Spalten + Ziel')
print(f'Neue train_df-Shape: {train_df.shape}')
print(f'Neue test_df-Shape: {test_df.shape}')
print('Benötigte Padding-Spalten in test_df:', missing_in_test or 'Keine')

Feature-Satz (train): 59 Spalten + Ziel
Neue train_df-Shape: (72878, 60)
Neue test_df-Shape: (50000, 59)
Benötigte Padding-Spalten in test_df: Keine


## Export der Feature-Sets
Nur die finalen, überarbeiteten DataFrames werden persistiert, sodass Notebook 05 direkt darauf aufbauen kann.

In [7]:
train_path = '../data/train/train_fe.csv'
test_path = '../data/test/test_fe.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f'Gespeichert: Train {train_df.shape}')
print(f'Gespeichert: Test {test_df.shape}')
print(f"Verhältnis zwischen Trainings- und Testdaten: {round(train_df.shape[0]/(train_df.shape[0]+test_df.shape[0])*100)}% Trainingsdaten")

Gespeichert: Train (72878, 60)
Gespeichert: Test (50000, 59)
Verhältnis zwischen Trainings- und Testdaten: 59% Trainingsdaten


## Fazit & Nächste Schritte
- Überflüssige Identifikatoren (`ID`, `Customer_ID`, `Name`, `SSN`, Textlisten) wurden entfernt oder in strukturierte Merkmale überführt.
- Hinzugefügte Features fokussieren auf Kredit-Historie, Zahlungsdisziplin, saisonale Muster sowie Belastungsquoten.
- Train- und Test-Daten teilen sich nun exakt dieselben Spalten (Speicherort: `data/train_fe.csv` & `data/test_fe.csv`) und können direkt im Notebook 05 modelliert werden.